# Linear Regression, Ridge, and Lasso

Fit and compare linear regression models with quadratic and sparse penalties. Keep training and test data separate while inspecting coefficient paths and prediction errors, so that improved fit is not confused with improved generalization.

This historical variant is retained for existing links. The main version is [Linear Regression, Ridge, and Lasso](ml_2_regression.ipynb).


## Run this tour

Run the cells in order with a Python 3 kernel. The first cell locates the companion data and toolbox and installs missing dependencies when needed. All worked examples include their implementation directly in this notebook. Random seeds make comparisons reproducible; you can change them to explore other samples.


In [ ]:
# Locate the companion toolbox locally, or fetch it for a standalone/Colab copy.
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

working = Path.cwd()
candidates = [working, working / "python", working.parent / "python"]
python_dir = next((p for p in candidates if (p / "nt_toolbox").is_dir()), None)
if python_dir is None:
    checkout = working / "numerical-tours-support"
    if not checkout.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "master",
                "https://github.com/gpeyre/numerical-tours.git",
                str(checkout),
            ],
            check=True,
        )
    python_dir = checkout / "python"
os.chdir(python_dir)
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))
requirements = python_dir / "requirements.txt"
if any(
    importlib.util.find_spec(name) is None
    for name in [
        "numpy",
        "scipy",
        "matplotlib",
        "skimage",
        "sklearn",
        "pywt",
        "ipywidgets",
        "cvxpy",
        "skfmm",
        "autograd",
        "progressbar",
        "celer",
    ]
):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True
    )

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams.update(
    {
        "figure.figsize": (8, 4),
        "figure.dpi": 100,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
        "image.cmap": "gray",
    }
)
%matplotlib inline


$\newcommand{\dotp}[2]{\langle #1, #2 \rangle}$
$\newcommand{\enscond}[2]{\lbrace #1, #2 \rbrace}$
$\newcommand{\pd}[2]{ \frac{ \partial #1}{\partial #2} }$
$\newcommand{\umin}[1]{\underset{#1}{\min}\;}$
$\newcommand{\umax}[1]{\underset{#1}{\max}\;}$
$\newcommand{\uargmin}[1]{\underset{#1}{argmin}\;}$
$\newcommand{\norm}[1]{\|#1\|}$
$\newcommand{\abs}[1]{\left|#1\right|}$
$\newcommand{\choice}[1]{ \left\{  \begin{array}{l} #1 \end{array} \right. }$
$\newcommand{\pa}[1]{\left(#1\right)}$
$\newcommand{\diag}[1]{{diag}\left( #1 \right)}$
$\newcommand{\qandq}{\quad\text{and}\quad}$
$\newcommand{\qwhereq}{\quad\text{where}\quad}$
$\newcommand{\qifq}{ \quad \text{if} \quad }$
$\newcommand{\qarrq}{ \quad \Longrightarrow \quad }$
$\newcommand{\ZZ}{\mathbb{Z}}$
$\newcommand{\CC}{\mathbb{C}}$
$\newcommand{\RR}{\mathbb{R}}$
$\newcommand{\EE}{\mathbb{E}}$
$\newcommand{\Zz}{\mathcal{Z}}$
$\newcommand{\Ww}{\mathcal{W}}$
$\newcommand{\Vv}{\mathcal{V}}$
$\newcommand{\Nn}{\mathcal{N}}$
$\newcommand{\NN}{\mathcal{N}}$
$\newcommand{\Hh}{\mathcal{H}}$
$\newcommand{\Bb}{\mathcal{B}}$
$\newcommand{\Ee}{\mathcal{E}}$
$\newcommand{\Cc}{\mathcal{C}}$
$\newcommand{\Gg}{\mathcal{G}}$
$\newcommand{\Ss}{\mathcal{S}}$
$\newcommand{\Pp}{\mathcal{P}}$
$\newcommand{\Ff}{\mathcal{F}}$
$\newcommand{\Xx}{\mathcal{X}}$
$\newcommand{\Mm}{\mathcal{M}}$
$\newcommand{\Ii}{\mathcal{I}}$
$\newcommand{\Dd}{\mathcal{D}}$
$\newcommand{\Ll}{\mathcal{L}}$
$\newcommand{\Tt}{\mathcal{T}}$
$\newcommand{\si}{\sigma}$
$\newcommand{\al}{\alpha}$
$\newcommand{\la}{\lambda}$
$\newcommand{\ga}{\gamma}$
$\newcommand{\Ga}{\Gamma}$
$\newcommand{\La}{\Lambda}$
$\newcommand{\Si}{\Sigma}$
$\newcommand{\be}{\beta}$
$\newcommand{\de}{\delta}$
$\newcommand{\De}{\Delta}$
$\newcommand{\phi}{\varphi}$
$\newcommand{\th}{\theta}$
$\newcommand{\om}{\omega}$
$\newcommand{\Om}{\Omega}$
$\newcommand{\eqdef}{\equiv}$


This tour studies linear regression method in conjunction with
regularization.
It contrasts ridge regression and the Lasso.


We recommend that after doing this Numerical Tours, you apply it to your
own data, for instance using a dataset from <https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/ LibSVM>.

_Disclaimer:_ these machine learning tours are intended to be
overly-simplistic implementations and applications of baseline machine learning methods.
For more advanced uses and implementations, we recommend
to use a state-of-the-art library, the most well known being
<http://scikit-learn.org/ Scikit-Learn>


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

Usefull functions to convert to a column/row vectors.


In [ ]:
#  convert to a column vector
def MakeCol(y):
    return y.reshape(-1, 1)


#  convert to a row vector
def MakeRow(y):
    return y.reshape(1, -1)


# find non zero/true elements
def find(x):
    return np.nonzero(x)[0]

Dataset Loading
---------------
We test the method on the prostate dataset in $n=97$ samples with
features $x_i \in \RR^p$ in dimension $p=8$. The goal is to predict the price value
$y_i \in \RR$.


Load the dataset.


In [ ]:
from scipy import io

name = "prostate"
U = io.loadmat("nt_toolbox/data/ml-" + name)
A = U["A"]
class_names = U["class_names"]

Randomly permute it.


In [ ]:
A = A[np.random.permutation(A.shape[0]), :]

Separate the features $X$ from the data $y$ to predict information.


In [ ]:
X = A[:, 0:-2]
y = MakeCol(A[:, -2])
c = MakeCol(A[:, -1])

$n$ is the number of samples, $p$ is the dimensionality of the features,


In [ ]:
[n, p] = X.shape
print(n, p)

Split into training and testing.


In [ ]:
I0 = find(c == 1)  # train
I1 = find(c == 0)  # test
n0 = I0.size
n1 = n - n0
X0 = X[I0, :]
y0 = y[I0]
X1 = X[I1, :]
y1 = y[I1]

Normalize the features by the mean and std of the *training* set.
This is optional.


In [ ]:
mX0 = X0.mean(axis=0)
sX0 = X0.std(axis=0)
X0 = (X0 - mX0) / sX0
X1 = (X1 - mX0) / sX0

Remove the mean (computed from the *test* set) to avoid introducing a bias term and a constant regressor.
This is optional.


In [ ]:
m0 = y0.mean()
y0 = y0 - m0
y1 = y1 - m0

# Use consistent names for the training and test matrices below.
A = X0.copy()
A1 = X1.copy()
y = y0.copy()
n = len(y)

Dimenionality Reduction and PCA
-------------------------------
In order to display in 2-D or 3-D the data, dimensionality is needed.
The simplest method is the principal component analysis, which perform an
orthogonal linear projection on the principal axsis (eigenvector) of the
covariance matrix.


Display the covariance matrix of the training set.


In [ ]:
C = X0.transpose().dot(X0)
plt.imshow(C);

In [ ]:
u = X0.transpose().dot(y0)
plt.clf()
plt.bar(np.arange(1, p + 1), u.flatten())
plt.axis("tight");

Compute PCA ortho-basis and
the feature in the PCA basis.


In [ ]:
U, s, V = np.linalg.svd(X0)
X0r = X0.dot(V.transpose())

Plot sqrt of the eigenvalues.


In [ ]:
plt.plot(s, ".-");

Display the features.


In [ ]:
pmax = min(p, 8)
k = 0
plt.clf()
for i in np.arange(0, pmax):
    for j in np.arange(0, pmax):
        k = k + 1
        plt.subplot(pmax, pmax, k)
        if i == j:
            plt.hist(X0[:, i], 6)
            plt.axis("tight")
        else:
            plt.plot(X0[:, j], X0[:, i], ".")
        plt.axis("tight")
        if i == 1:
            plt.title(class_names[0][j][0])
        plt.tick_params(axis="x", labelbottom=False)
        plt.tick_params(axis="y", labelleft=False)

Display the points cloud of feature vectors in 2-D PCA space.


In [ ]:
plt.plot(X0r[:, 0], X0r[:, 1], ".")
plt.axis("equal");

1D plot of the function to regress along the main eigenvector axes.


In [ ]:
plt.clf()
for i in np.arange(0, 3):
    plt.subplot(3, 1, i + 1)
    plt.plot(X0r[:, i], y0, ".")
    plt.axis("tight")

Linear Regression
-----------------
We look for a linear relationship
  $ y_i = \dotp{w}{x_i} $
written in matrix format
  $ y= X w $
where the rows of $X \in \RR^{n \times p}$ stores the features $x_i \in \RR^p$.


Since here $ n > p $, this is an over-determined system, which can
solved in the least square sense
  $$ \umin{ w }  \norm{Xw-y}^2 $$
whose solution is given using the Moore-Penrose pseudo-inverse
  $$ w = (X^\top X)^{-1} X^\top y $$


Least square solution.


In [ ]:
w = np.linalg.solve(X0.transpose().dot(X0), X0.transpose().dot(y0))

Prediction (along 1st eigenvector).


In [ ]:
plt.clf()
plt.plot(X1.dot(w), ".-")
plt.plot(y1, ".-")
plt.axis("tight")
plt.legend(("$y_1$", "$X_1 w$"));

Mean-square error on testing set.


In [ ]:
E = np.linalg.norm(X1.dot(w) - y1) / np.linalg.norm(y1)
print(("Relative prediction error: " + str(E)));

Although this is not an effective method to solve this problem (a more efficient approach is to use for instance conjugate gradient), one can do a gradient descent to minimize the function
$$ 
    \min_w J(w) = \frac{1}{2}\norm{X w-y}^2.
$$


In [ ]:
def J(w):
    return 1 / 2 * np.linalg.norm(X0.dot(w) - y0) ** 2

The gradient of $J$ is
$$
    \nabla J(w) = X^\top (X w - y).
$$


In [ ]:
def GradJ(w):
    return X0.transpose().dot(X0.dot(w) - y0)

The maxium step size allowable by the gradient descent is 
$$
    \tau \leq \tau_\max \eqdef \frac{2}{\norm{XX^\top}_{op}}
$$
where $\norm{\cdot}_{op}$ is the maximum singular eigenvalue.


In [ ]:
tau = 1 / np.linalg.norm(X0, 2) ** 2

Initialize the algorithm to $w=0$.


In [ ]:
w = np.zeros((p, 1))

One step of gradient descent reads:
$$  w \leftarrow w - \tau \nabla J(w). $$


In [ ]:
w = w - tau * GradJ(w)

In [ ]:
tau_mult = [0.1, 0.5, 1, 1.8]

__Worked example 0__

Display the evolution of the training error $J(w)$ as a function of the number of iterations.
Test for diffetent values of $\tau$.


In [ ]:
def f(x):
    return 0.5 * np.linalg.norm(A @ x - y) ** 2


def Gradf(x):
    return A.T @ (A @ x - y)


niter = 100
flist = np.zeros((niter, 1))

tau_mult = [0.1, 0.5, 1, 1.8, 1.98]
xopt = np.linalg.solve(A.transpose().dot(A), A.transpose().dot(y))
plt.clf()
fig, (ax1, ax2) = plt.subplots(2, 1)
for itau in np.arange(0, 5):
    tau = tau_mult[itau] / np.linalg.norm(A, 2) ** 2
    x = np.zeros((p, 1))
    for i in np.arange(0, niter):
        flist[i] = f(x)
        x = x - tau * Gradf(x)
    # plt.subplot(2,1,1)
    ax1.plot(flist)
    ax1.axis("tight")
    plt.title("f(x_k)")
    # plt.subplot(2,1,2)
    e = np.log10(flist - f(xopt) + 1e-20)
    ax2.plot(e - e[0], label=str(tau_mult[itau]))
    ax2.axis("tight")
    leg = ax2.legend()
    # ax2.legend( str( tau_mult[itau] ) )
    plt.title("$log(f(x_k)-min J)$")

The optimal step size to minimize a quadratic function $\dotp{C w}{w}$ is
$$
\tau_{\text{opt}} = \frac{2}{\sigma_\min(C) + \sigma_\max(C)}
$$


In [ ]:
C = X0.transpose().dot(X0)
tau_opt = 2 / (np.linalg.norm(C, 2) + np.linalg.norm(C, -2))
print(
    ("Optimal tau = " + str(tau_opt * np.linalg.norm(X0, 2) ** 2)) + " / |X_0 X_0^T|"
);

Stochastic Gradient Method
=======

We use SGD (which is *not* actually a descent algorithm) to minimize the quadratic risk
$$ \umin{w} J(w) \eqdef \frac{1}{n} \sum_{i=1}^n f_i(w) = \frac{1}{n} \sum_{i=1}^n \frac{1}{2} ( \dotp{w}{x_i}-y_i )^2 $$
where we used
$$ f_i(w) \eqdef \frac{1}{2} ( \dotp{w}{x_i}-y_i )^2$$
The algorithm reads
$$ w_{k+1} = w_k - \tau_k \nabla f_{i_k}(w_k)$$
where at each iteration $i_k$ is drawn in $\{1,\ldots,n\}$ uniformly at random.


In [ ]:
w = np.zeros((p, 1))

Draw $i_k$ are random.


In [ ]:
ik = int(np.floor(np.random.rand() * n0))

Compute $\nabla f_{i_k}(w_k)$.


In [ ]:
gk = (X0[ik, :].dot(w) - y0[ik]) * X0[ik, :].transpose()

Set the step size $w_k$ (for convergence is should converge to 0) and perform the update.


In [ ]:
tauk = 1 / np.linalg.norm(X0, 2) ** 2
w = w - tauk * gk

Test different *fixed* step size $\tau_k=\tau$.


In [ ]:
wopt = np.linalg.solve(
    X0.transpose().dot(X0), X0.transpose().dot(y0)
)  # least square solution
niter = 5000
Jlist = np.zeros((niter, 1))
tau_mult = [0.05, 0.3, 0.8]
for itau in np.arange(0, len(tau_mult)):
    tauk = tau_mult[itau] / np.linalg.norm(X0, 2) ** 2
    w = np.zeros((p, 1))
    for i in np.arange(0, niter):
        ik = int(np.floor(np.random.rand() * n0))
        gk = (X0[ik, :].dot(w) - y0[ik]) * X0[ik, :].transpose()  # stochastic gradient
        # gk = X0.transpose().dot( X0.dot(w)-y0 )  # batch gradient
        w = w - tauk * gk
        Jlist[i] = J(w)
    plt.plot(Jlist / J(wopt) - 1)

Average on different runs to see the impact of the step size.


In [ ]:
niter = 8000
tau_mult = [0.05, 0.8]
nruns = 10  # number of runs to compute the average performance
for itau in np.arange(0, len(tau_mult)):
    tauk = tau_mult[itau] / np.linalg.norm(X0, 2) ** 2
    Jlist = np.zeros((niter, 1))
    for iruns in np.arange(0, nruns):
        w = np.zeros((p, 1))
        for i in np.arange(0, niter):
            ik = int(np.floor(np.random.rand() * n0))
            gk = (X0[ik, :].dot(w) - y0[ik]) * X0[ik, :].transpose()
            w = w - tauk * gk
            Jlist[i] = Jlist[i] + J(w)
    plt.plot((Jlist / nruns) / J(wopt) - 1)

Use a decaying step size $\tau_k=\frac{\tau_0}{1+k/k_0}$.


In [ ]:
niter = 20000
Jlist = np.zeros((niter, 1))
w = np.zeros((p, 1))
for i in np.arange(0, niter):
    tauk = 1 / np.linalg.norm(X0, 2) ** 2 * 1 / (1 + i / 10)
    ik = int(np.floor(np.random.rand() * n0))
    gk = (X0[ik, :].dot(w) - y0[ik]) * X0[ik, :].transpose()
    w = w - tauk * gk
    Jlist[i] = J(w)
plt.plot(Jlist / J(wopt) - 1)
print(Jlist[-1] / J(wopt) - 1)

Use a decaying step size $\tau_k=\frac{\tau_0}{1+\sqrt{k/k_0}}$ and average the iteration $\frac{1}{K}\sum_{k<K}w_k$.


In [ ]:
niter = 20000
Jlist = np.zeros((niter, 1))
w = np.zeros((p, 1))
w1 = np.zeros((p, 1))
for i in np.arange(0, niter):
    tauk = 1 / np.linalg.norm(X0, 2) ** 2 * 1 / (1 + np.sqrt(i / 10.0))
    ik = int(np.floor(np.random.rand() * n0))
    gk = (X0[ik, :].dot(w) - y0[ik]) * X0[ik, :].transpose()
    w = w - tauk * gk
    w1 = 1 / (i + 1) * w + i / (i + 1) * w1
    Jlist[i] = J(w1)
plt.plot(Jlist / J(wopt) - 1);

Ridge Regularization
=======

Regularization is obtained by introducing a penalty. It is often called
ridge regression, and is defined as
  $$ \umin{ w }  \norm{Xw-y}^2 + \lambda \norm{w}^2 $$
where $\lambda>0$ is the regularization parameter.


The solution is given using the following equivalent formula
  $$ w = (X^\top X + \lambda \text{Id}_p )^{-1} X^\top y, $$
  $$ w = X^\top ( XX^\top + \lambda \text{Id}_n)^{-1} y, $$
When $p<n$ (which is the case here), the first formula should be
prefered.


In contrast, when the dimensionality $p$ of the feature is very
large and there is little data, the second is faster. Furthermore, this
second expression is generalizable to Kernel Hilbert space setting,
corresponding possibly to $p=+\infty$ for some kernels.


In [ ]:
Lambda = 0.2 * np.linalg.norm(X0) ** 2
w = np.linalg.solve(X0.transpose().dot(X0) + Lambda * np.eye(p), X0.transpose().dot(y0))
u = np.linalg.solve(X0.dot(X0.transpose()) + Lambda * np.eye(n0), y0)
w1 = X0.transpose().dot(u)
print(("Error (should be 0): " + str(np.linalg.norm(w - w1) / np.linalg.norm(w))))

__Worked example 1__

Display the evolution of the test error $E$ as a function of $\lambda$.


In [ ]:
q = 50
lmax = np.linalg.norm(A, 2) ** 2
lambda_list = lmax * np.linspace(0.3, 1e-3, q)
X = np.zeros((p, q))
E = np.zeros((q, 1))
for i in np.arange(0, q):
    Lambda = lambda_list[i]
    x = np.linalg.solve(A.transpose().dot(A) + Lambda * np.eye(p), A.transpose().dot(y))
    X[:, i] = x.flatten()  # bookkeeping
    E[i] = np.linalg.norm(A1.dot(x) - y1) / np.linalg.norm(y1)
# find optimal lambda
i = E.argmin()
lambda0 = lambda_list[i]
xRidge = X[:, i]
print("Ridge: " + str(E.min() * 100) + "%")
# Display error evolution.
plt.clf()
plt.plot(lambda_list / lmax, E)
plt.plot([lambda0 / lmax, lambda0 / lmax], [E.min(), E.max()], "r--")
plt.axis("tight")
plt.xlabel(r"$\lambda/|X|^2$")
plt.ylabel("$E$")

__Worked example 2__

Display the regularization path, i.e. the evolution of $w$ as a function
of $\lambda$.


In [ ]:
plt.clf()
for i in np.arange(0, p):
    plt.plot(lambda_list / lmax, X[i, :], label=class_names[0][i])
plt.plot(
    [lambda0 / lmax, lambda0 / lmax], [X.flatten().min(), X.flatten().max()], "r--"
)
plt.axis("tight")
plt.xlabel(r"$\lambda/|X|^2$")
plt.ylabel("$x_i$")
plt.legend()

Sparse Regularization
---------------------
In order to perform feature selection (i.e. select a subsect of the
features which are the most predictive), one needs to replace the
$\ell^2$ regularization penalty by a sparsity inducing regularizer. The
most well known is the $\ell^1$ norm
$$ \norm{w}_1 \eqdef \sum_i \abs{w_i} . $$


The energy to minimize is
$$ \umin{w} J(w) \eqdef \frac{1}{2}\norm{X w-y}^2 + \lambda \norm{w}_1. $$


In [ ]:
def J(w, Lambda):
    return 1 / 2 * np.linalg.norm(X0.dot(w) - y0) ** 2 + Lambda * np.linalg.norm(w, 1)

The simplest iterative algorithm to perform the minimization is the
so-called iterative soft thresholding (ISTA), aka proximal gradient aka
forward-backward.


It performs first a gradient step (forward) of the smooth part $\frac{1}{2}\norm{X w-y}^2$ of the
functional and then a proximal step (backward) step which account for the
$\ell^1$ penalty and induce sparsity. This proximal step is the soft-thresholding operator
$$ \Ss_s(x) \eqdef \max( \abs{x}-\lambda,0 ) \text{sign}(x).  $$


In [ ]:
def Soft(x, s):
    return np.maximum(abs(x) - s, np.zeros(x.shape)) * np.sign(x)

The ISTA algorithm reads
$$ w_{k+1} \eqdef \Ss_{\la\tau}( w_k - \tau X^\top ( X w_k - y )  ), $$
where, to ensure convergence, the step size should verify $ 0 < \tau <
2/\norm{X}^2  $ where $\norm{X}$ is the operator norm.


Display the soft thresholding operator.


In [ ]:
t = np.linspace(-5, 5, 201)
plt.clf()
plt.plot(t, Soft(t, 2))
plt.axis("tight");

Descent step size.


In [ ]:
tau = 1.5 / np.linalg.norm(X0, 2) ** 2

Choose a regularization parameter $\la$.


In [ ]:
lmax = abs(X0.transpose().dot(y0)).max()
Lambda = lmax / 10

Initialization $w_0$.


In [ ]:
w = np.zeros((p, 1))

A single ISTA step.


In [ ]:
C = X0.transpose().dot(X0)
u = X0.transpose().dot(y0)


def ISTA(w, Lambda, tau):
    return Soft(w - tau * (C.dot(w) - u), Lambda * tau)


w = ISTA(w, Lambda, tau)

__Worked example 3__

Implement the ISTA algorithm, display the convergence of the energy.


In [ ]:
def f(x, Lambda):
    return 0.5 * np.linalg.norm(A @ x - y) ** 2 + Lambda * np.sum(np.abs(x))


niter = 400
flist = np.zeros((niter, 1))
x = np.zeros((p, 1))
for i in np.arange(0, niter):
    flist[i] = f(x, Lambda)
    x = ISTA(x, Lambda, tau)
ndisp = int(niter / 4)

plt.clf()
plt.subplot(2, 1, 1)
plt.plot(flist[0:ndisp])
plt.axis("tight")
plt.title("f(x_k)")
plt.subplot(2, 1, 2)
e = np.log10(flist[0:ndisp] - flist.min() + 1e-20)
plt.plot(e - e[0])
plt.axis("tight")
plt.title("$log(f(x_k)-min f)$")

__Worked example 4__

Compute the test error along the full regularization path. You can start by large $\lambda$ and use a warm restart procedure
to reduce the computation time. Compute the classification error.
ind optimal lambda
isplay error evolution.


In [ ]:
q = 200
lambda_list = lmax * np.linspace(0.6, 1e-3, q)
X = np.zeros((p, q))
E = np.zeros((q, 1))
x = np.zeros((p, 1))
niter = 500
for iq in np.arange(0, q):
    Lambda = lambda_list[iq]
    # ISTA #
    for i in np.arange(0, niter):
        x = ISTA(x, Lambda, tau)
    X[:, iq] = x.flatten()  # bookkeeping
    E[iq] = np.linalg.norm(A1.dot(x) - y1) / np.linalg.norm(y1)
# find optimal Lambda
i = E.argmin()
lambda0 = lambda_list[i]
xSparse = X[:, i]
print("Lasso: " + str(E.min() * 100) + "%")
# Display error evolution.
plt.clf()
plt.plot(lambda_list / lmax, E)
plt.plot([lambda0 / lmax, lambda0 / lmax], [E.min(), E.max()], "r--")
plt.axis("tight")
plt.xlabel(r"$\lambda/|A^* y|_\infty$")
plt.ylabel("$E$")

__Worked example 5__

Display the regularization path, i.e. the evolution of $w$ as a function
of $\lambda$.
lot(lambda_list, W', 'LineWidth', 2);


In [ ]:
plt.clf()
for i in np.arange(0, p):
    plt.plot(lambda_list / lmax, X[i, :], label=class_names[0][i])
plt.plot(
    [lambda0 / lmax, lambda0 / lmax], [X.flatten().min(), X.flatten().max()], "r--"
)
plt.axis("tight")
plt.xlabel(r"$\lambda/|A^* y|_\infty$")
plt.ylabel("$x_i$")
plt.legend()

__Worked example 6__

Compare the optimal weights for ridge and lasso.


In [ ]:
plt.clf()
plt.bar(np.arange(1, p + 1), abs(xSparse))
plt.bar(np.arange(1, p + 1), -abs(xRidge))
plt.legend(("Lasso", "Ridge"))

<script>
  $(document).ready(function(){
      $('div.prompt').hide();
  });
</script>


## References and further reading

- Trevor Hastie, Robert Tibshirani, and Jerome Friedman. [The Elements of Statistical Learning](https://hastie.su.domains/ElemStatLearn/). 2009, 2nd ed., Springer. Regression, classification, regularization, and model assessment.

- Gilbert Strang. [Linear Algebra and Learning from Data](https://math.mit.edu/~gs/learningfromdata/). 2019, Wellesley-Cambridge Press. Matrix factorizations, least squares, and low-rank representations.

- Fabian Pedregosa et al.. [Scikit-learn: Machine Learning in Python](https://www.jmlr.org/papers/v12/pedregosa11a.html). 2011, Journal of Machine Learning Research 12, 2825–2830. Practical estimators and reproducible model evaluation.

- Stephen Boyd and Lieven Vandenberghe. [Convex Optimization](https://web.stanford.edu/~boyd/cvxbook/). 2004, Cambridge University Press. Convexity, duality, optimality conditions, and interior-point methods.

- Robert Tibshirani. [Regression Shrinkage and Selection via the Lasso](https://doi.org/10.1111/j.2517-6161.1996.tb02080.x). 1996, Journal of the Royal Statistical Society B 58(1), 267–288. Sparse regression through an absolute-value penalty.
